In [2]:
import pandas as pd
import re

INPUT_CSV = "Dataset.csv"
X_OUT = "x_values.csv"
Y_OUT = "y_values.csv"
MAP_OUT = "label_map.csv"
Y_DEBUG_OUT = "y_values_debug.csv"

feature_cols = [
    "I_rms_A", "I_max_A",
    "Amp_50Hz", "Amp_100Hz", "Amp_150Hz", "Amp_200Hz", "Amp_250Hz"
]

label_to_id = {
    "noDevice": 0,
    "loetstation": 1,
    "ventilator": 2,
    "laptobNt": 3,
}

DROP_BAD_ROWS = True
PRINT_BAD_ROWS = True
DROP_UNKNOWN_LABELS = False  # True: unbekannte Labels entfernen | False: abbrechen

def normalize_label(s: str) -> str:
    s = str(s)
    s = s.replace("\ufeff", "")   # BOM
    s = s.replace("\u00a0", " ")  # NBSP -> normales Space
    s = s.replace("\r", "")       # CR raus
    s = s.strip()
    # mehrere Whitespaces zusammenfassen
    s = re.sub(r"\s+", " ", s)
    return s

df = pd.read_csv(INPUT_CSV)

missing = [c for c in feature_cols + ["label"] if c not in df.columns]
if missing:
    raise ValueError(f"Fehlende Spalten in CSV: {missing}")

# Labels normalisieren
df["label"] = df["label"].apply(normalize_label)

# Unbekannte Labels prüfen (und mit repr anzeigen!)
unknown_labels = sorted(set(df["label"].unique()) - set(label_to_id.keys()))
if unknown_labels:
    print("⚠️ Unbekannte Labels gefunden (mit repr, damit man Sonderzeichen sieht):")
    for u in unknown_labels:
        print("  ", repr(u))

    if DROP_UNKNOWN_LABELS:
        df = df[~df["label"].isin(unknown_labels)].copy()
        print(f"✅ Unbekannte Labels entfernt. Übrig: {len(df)} Zeilen")
    else:
        raise ValueError(
            f"Unbekannte Labels gefunden: {unknown_labels}\n"
            f"Erlaubt sind nur: {sorted(label_to_id.keys())}"
        )

# Features numerisch machen
X_num = df[feature_cols].apply(pd.to_numeric, errors="coerce")
bad_feat = X_num.isna().any(axis=1)
bad_label = df["label"].isna() | (df["label"].str.len() == 0)
bad = bad_feat | bad_label
bad_count = int(bad.sum())

if bad_count > 0:
    print(f"⚠️ Gefunden: {bad_count} fehlerhafte Zeilen (NaN/leer)")

    if PRINT_BAD_ROWS:
        print("Indices:", df.index[bad].tolist())
        print(df.loc[bad, feature_cols + ["label"]])

    if DROP_BAD_ROWS:
        df = df.loc[~bad].copy()
        X_num = X_num.loc[~bad].copy()
        print(f"✅ Schlechte Zeilen entfernt. Übrig: {len(df)} Zeilen")
    else:
        raise ValueError("Abbruch wegen fehlerhaften Zeilen (setze DROP_BAD_ROWS=True zum Entfernen).")

# X schreiben
X = X_num.astype("float32")
X.to_csv(X_OUT, index=False)

# y schreiben
y = df["label"].map(label_to_id).astype("int32")
pd.DataFrame({"y": y}).to_csv(Y_OUT, index=False)

# Debug
pd.DataFrame({"y": y, "label": df["label"]}).to_csv(Y_DEBUG_OUT, index=False)

# Mapping schreiben
pd.DataFrame({"label": list(label_to_id.keys()), "id": list(label_to_id.values())}).to_csv(MAP_OUT, index=False)

print("✅ Fertig")
print("X:", X_OUT, "shape =", X.shape)
print("y:", Y_OUT, "shape =", y.shape)
print("y debug:", Y_DEBUG_OUT)
print("Mapping:", MAP_OUT)
print("Label->ID:", label_to_id)


✅ Fertig
X: x_values.csv shape = (80, 7)
y: y_values.csv shape = (80,)
y debug: y_values_debug.csv
Mapping: label_map.csv
Label->ID: {'noDevice': 0, 'loetstation': 1, 'ventilator': 2, 'laptobNt': 3}
